In [8]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

sys.path.insert(0, os.path.abspath("../src"))

from train import load_model
from predict import predict, get_shap_explanation



In [ ]:
pipeline, thresholds = load_model()
threshold = thresholds["recall_threshold"]

X_test = pd.read_csv("../data/X_test.csv")
y_test = pd.read_csv("../data/y_test.csv").squeeze()
X_train = pd.read_csv("../data/X_train.csv")

print(f"Recall threshold: {threshold}")
print(f"Test set size: {len(X_test)} customers")

In [ ]:
# Manually built customer with features of a high risk customer, i.e. month-to-month contract, fibre optic, high charges, short tenure

customer = {col: 0 for col in X_train.columns} # fill dict with 0s of len X_train

customer.update({
    "gender": 1,
    "senior citizen": 0,
    "partner": 0,
    "dependents": 0,
    "phone service": 1,
    "multiple lines": 0,
    "online security": 0,
    "online backup": 0,
    "device protection": 0,
    "tech support": 0,
    "streaming tv": 1,
    "streaming movies": 1,
    "paperless billing": 1,
    "contract": 0,                              # month to month contract plan
    "monthly charges": 85.0,
    "total charges": 340.0,
    "number of subscriptions": 3,
    "average charge per subscription": 28.3,
    "internet service_Fiber optic": 1,
    "payment method_Electronic check": 1,
    "tenure month type_0 to 12": 1,             # short tenure
})

customer_row = pd.DataFrame([customer])[X_train.columns]
print("Customer profile built successfully.")
display(customer_row.style.hide(axis="index"))

In [ ]:
result = predict(pipeline, X_train, customer_row, threshold=threshold)

probability = result["probability"]
prediction = result["prediction"]

print(f"Churn probability:  {probability:.1%}")
print(f"Prediction:         {'Churned' if prediction == 1 else 'Retained'}")
print(f"Threshold used:     {threshold}")

In [ ]:
explanation = result["explanation"]

shap.plots.waterfall(
    shap.Explanation(
        values=np.array(explanation["shap values"]),
        base_values=explanation["base value"],
        data=np.array(explanation["feature values"]),
        feature_names=explanation["feature names"]
    ),
    show=False
)

plt.title("Why is this customer predicted to churn?")
plt.tight_layout()
plt.show()

In [ ]:
# Low-risk customer profile:
# Two-year contract, DSL, low charges, long tenure

low_risk_customer = {col: 0 for col in X_train.columns}

low_risk_customer.update({
    "gender": 0,
    "senior citizen": 0,
    "partner": 1,
    "dependents": 1,
    "phone service": 1,
    "multiple lines": 0,
    "online security": 1,
    "online backup": 1,
    "device protection": 0,
    "tech support": 1,
    "streaming tv": 0,
    "streaming movies": 0,
    "paperless billing": 0,
    "contract": 2,                   # 2 year
    "monthly charges": 45.0,
    "total charges": 3200.0,
    "number of subscriptions": 4,
    "average charge per subscription": 11.25,
    "internet service_DSL": 1,
    "payment method_Mailed check": 1,
    "tenure month type_48 to 72": 1,            # long tenure
})

low_risk_row = pd.DataFrame([low_risk_customer])[X_train.columns]
low_risk_result = predict(pipeline, X_train, low_risk_row, threshold=threshold)

print(f"Churn probability:  {low_risk_result['probability']:.1%}")
print(f"Prediction:         {'Churned' if low_risk_result['prediction'] == 1 else 'Retained'}")

In [ ]:
low_risk_explanation = low_risk_result["explanation"]

shap.plots.waterfall(
    shap.Explanation(
        values=np.array(low_risk_explanation["shap values"]),
        base_values=low_risk_explanation["base value"],
        data=np.array(low_risk_explanation["feature values"]),
        feature_names=low_risk_explanation["feature names"]
    ),
    show=False
)

plt.title("Why is this customer predicted to stay?")
plt.tight_layout()
plt.show()

## Demonstration - Customer Churn Prediction

This notebook demonstrates the trained logistic regression model on two manually constructed customer profiles:

- **High-risk customer**: month-to-month contract, fiber optic internet, high monthly charges, short tenure
- **Low-risk customer**: two-year contract, DSL internet, low monthly charges, long tenure

For each customer, the model outputs a churn probability and a SHAP waterfall chart explaining which features drove the prediction.

Model and threshold selected from `05_model_tuning.ipynb`. Full analysis in `03_modelling.ipynb` and `04_shap.ipynb`.